**Bioseñales y sistemas: Proyecto 1**

Daniel Lozano<br>
Samuel Ochoa<br>
Alejandro Restrepo

Este primer proyecto se centra en la utilización de herramientas de Python para la manipulación y análisis de señales de Electroencefalografía (EEG) adquiridas por repositorios libres encontrados en bases de datos en internet como OpenNeuro. En el caso de este proyecto, se hace uso del Dataset "EEG Motor Movement/Imagery Dataset", originalmente creado por Gerwin Schalk y su equipo del programa de BCI R&D, Wadsworth Center, New York State Depaartment of Health, Albany, NY para la base de datos PhysioBank.

Aquí se busca evaluar la factibilidad de construir interfaces Cerebro-Computador (BCI) no invasivas mediante un conjunto de datos que registra la actividad cerebral en condiciones de movimiento real, imaginación del movimiento y estado de reposo. En base al contenido aprendido en el curso, se asume que el enfoque principal será realizar las comparaciones e inferencias estadisticas pertinentes sobre la Densidad Espectral de Potencia (PSD) en ritmos cerebrales específicos para diferenciar las condiciones. Esta última, no solo por ser tema central de la última unidad, sino porque se puede aprovechar sus aplicaciones para identificar periodicidades escondidas en una función de variable discreta, estimar la entropía de un proceso aleatorio o porque puede proporcionar información muy valiosa sobre la dinámica interna de muchos sistemas físicos [1].

Para hablar de la BCI, primero tenemos que entender el nacimiento de las neurociencias. En 1924, el neurólogo alemán Hans Berger, siguiendo el trabajo de Richard Caton (1875), descubre la actividad eléctrica presente en el cerebro, registrando la primera actividad mediante EEG. Mediante el análisis de este, consiguió clasificar los tipos de ondas cerebrales. No fue sino hasta 1973 que Jacques Vidal introduce el término BCI en la literatura científica mientras estaba en la Universidad de California, Los Ángeles (UCLA) planteando el uso de señales EEG para el control de dispositivos externos [2].

Hasta la fecha, se han visto avances sobre la idea de Vidal, principalmente con fines médicos, donde se ha buscado implantar prótesis neuronales para la recuperación de la visión, audición o movilidad. A día de hoy la mayor expectativa se encuentra en la empresa Neuralink, dado que ha sido líder indiscutible en el desarrollo de tecnología que permita la comunicación bidireccional entre dispositivos y el cerebro.

Los conceptos de coherencia y conectividad cerebral, en su conjunto, establecen cómo las redes neuronales se comunican para procesar información. La conectividad cerebral describe las redes de conexiones funcionales y anatómicas en todo el cerebro; la comunicación a través de dicha red depende de las oscilaciones neuronales. Por otro lado, la coherencia trata de un método matemático que permite determinar si dos o más sensores, o regiones cerebrales, presentan una actividad oscilatoria neuronal similar entre sí [3].

**Metodología y plan de análisis**

Antes de la implementación técnica, se definen los parámetros biológicos y los criterios de selección de datos:


**1. Electrodos de interés:** Se analizarán prioritariamente C3 (corteza motora izquierda, controla mano derecha), C4 (corteza motora derecha, controla mano izquierda) y Cz (línea media), debido a su ubicación sobre el homúnculo motor.
   
*   **Ritmos cerebrales:** 
    *   **Ritmo Mu (8-13 Hz):** Asociado a la planificación motora; su potencia disminuye durante la actividad activa (ERD) [4].

    * **Ritmo Beta (13-30 Hz):** Relacionado con el control y la ejecución del movimiento [4]. 

**2. Formulación de Hipótesis:** Se plantean las siguientes hipótesis para diferenciar las cuatro condiciones de estudio (movimiento real/imaginado de ambas manos y reposo):

* **Hipótesis 1 (ERD/ERS):** La ejecución o imaginación de movimiento generará una Desincronización Relacionada con Eventos (ERD), manifestada como una caída en la potencia de la banda Mu en el hemisferio contralateral al movimiento [4].

* **Hipótesis 2 (Lateralidad):** El índice de lateralidad de la potencia será un predictor de la tarea realizada (mano derecha vs. mano izquierda).

* **Hipótesis 3 (Coherencia):** Durante las tareas de imaginación activa, se presentará un aumento en la coherencia funcional entre los electrodos motores (C3, C4) comparado con el estado de reposo.

**Herramientas a usar:** Para probar las hipótesis, se utilizará Python con las librerías MNE-Python (procesamiento de EEG), SciPy (análisis espectral y filtros) y Pandas (manejo de datos y estadística descriptiva).

In [30]:
import mne
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import warnings

In [19]:
#Codigo para el filtrado de la señal
def filtro_MuBeta(raw):
    """La función recibe el archivo .raw y retorna las 2 señales filtradas, por cada banda.
    Mu (8-13 Hz) y Beta (14-30 Hz).
    """
    raw_mu = raw.copy().filter(
        l_freq = 8.0, h_freq = 12.0,
        method='fir', fir_window='hamming', phase='zero', verbose=False
    )
    raw_beta = raw.copy().filter(
        l_freq = 13.0, h_freq = 30.0,
        method='fir', fir_window='hamming', phase='zero', verbose=False
    )
    return raw_mu, raw_beta

In [ ]:
carpeta_datos = './data'
sujetos = sorted([
    f.replace('sub-', '')
    for f in os.listdir(carpeta_datos)
    if os.path.isdir(os.path.join(carpeta_datos, f)) and f.startswith('sub-')
])
print(sujetos)
runs_real = ['3', '7', '8']
runs_imaginado = ['4', '8', '12']
canales = ['C3', 'C4', 'Cz']

# Event IDs según la documentación del dataset
# T1 = mano izquierda, T2 = mano derecha (en runs de Task 1 y Task 2)
event_id = {'TASK1T1': 1, 'TASK1T2': 2}   # real
event_id_im= {'TASK2T1': 1, 'TASK2T2': 2} # imaginado

['001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017', '018', '019', '020', '021', '022', '023', '024', '025', '026', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '041', '042', '043', '044', '045', '046', '047', '048', '049', '050', '051', '052', '053', '054', '055', '056', '057', '058', '059', '060', '061', '062', '063', '064', '065', '066', '067', '068', '069', '070', '071', '072', '073', '074', '075', '076', '077', '078', '079', '080', '081', '082', '083', '084', '085', '086', '087', '088', '089', '090', '091', '092', '093', '094', '095', '096', '097', '098', '099', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109']


In [21]:
def procesar_raw(raw, sujeto, mostrar_fases):
    etapas = {"1_raw_original": raw.copy()}

    raw.pick(canales)
    etapas["2_canales_seleccionados"] = raw.copy()

    raw.set_eeg_reference("average", projection=False, verbose=False)
    etapas["3_rereferencia"] = raw.copy()

    if mostrar_fases:
        fig, axes = plt.subplots(1, len(etapas), figsize=(16, 4))
        fig.suptitle(f'Fases de Procesamiento   sujeto {sujeto}', fontsize=13)

        for ax, (nombre, r) in zip(axes, etapas.items()):
            r.pick([c for c in canales if c in r.ch_names])
            r.compute_psd(fmin=1, fmax=40).plot(axes=ax, show=False)
            ax.set_title(nombre.replace('_', ' '), fontsize=9)
            plt.tight_layout()
            plt.show()

    return raw

In [22]:
#Codigo pa calcular la PSD
def calcular_psd(epocas, fmin, fmax):
    espectro = epocas.compute_psd(method='welch', fmin=fmin, fmax=fmax, verbose=False)
    return espectro.get_data().mean(axis=(0,2)) #axis (0,2) porque se hace promedio
    #por epocas (0) y frecuencias (2)

In [ ]:
#Código para la carga y filtrado de los archivos

#Cada entrada es una fila por sujeto [C3, C4, Cz]
registros = {
    'real_izq': [], 'real_der': [],
    'imag_izq': []
}



               C3            C4            Cz
001  1.556228e-12  1.698783e-12  5.260674e-13
002  1.105898e-12  8.451084e-13  1.253777e-12
003  8.432910e-13  8.072049e-13  3.042132e-13
004  4.022648e-13  5.069427e-13  1.643920e-13
005  3.025574e-13  3.778541e-13  1.320057e-13
..            ...           ...           ...
105  3.634239e-13  3.825156e-13  1.352365e-13
106  1.542276e-12  1.422653e-12  9.602455e-13
107  5.691866e-13  5.612898e-13  1.823726e-13
108  2.555102e-12  2.163829e-12  5.713431e-13
109  7.514849e-11  2.994376e-10  7.579415e-11

[109 rows x 3 columns]


**Referencias**

[1] D. G. Manolakis y V. K. Ingle, Tratamiento digital de señales: principios, algoritmos y aplicaciones. México: Prentice Hall, 1997.<br>
[2] J. J. Vidal, “Toward direct brain–computer communication,” Annual Review of Biophysics and Bioengineering, vol. 2, pp. 157–180, 1973, doi: 10.1146/annurev.bb.02.060173.001105.<br>
[3] S. M. Bowyer, “Coherence: A measure of the brain networks: past and present,” Neuropsychiatric Electrophysiology, vol. 2, no. 1, 2016, doi: 10.1186/s40810-015-0015-7.<br>
[4] H. Yu, S. Ba, Y. Guo, L. Guo, y G. Xu, “Effects of motor imagery tasks on brain functional networks based on EEG mu/beta rhythm,” Brain Sciences, vol. 12, no. 2, p. 194, 2022, doi: 10.3390/brainsci12020194.